# BERTurk eğitimi (A2.1)

Girdi: `greenchoice export-nlp-data` çıktısı `train.csv`, `val.csv`, `test.csv`
(`data/interim/nlp_egitim/`). Çıktı: `berturk_v0.zip` → bilgisayarda `models/berturk_v0/` altına açılır,
sonra `greenchoice build --nlp berturk --nlp-model models/berturk_v0`.

- **Çalışma zamanı → T4 GPU.**
- Dosyalar yorum metni içerir: yalnızca bu oturuma yükleyin, Drive'a veya başka yere kopyalamayın.
  Oturum kapanınca silinir.
- Test seti **bir kez** kullanılır (son hücre); eşikler ve epoch seçimi yalnızca val ile yapılır.

In [ ]:
!pip -q install "transformers>=4.40" "torch>=2.2" scikit-learn pandas

In [ ]:
from google.colab import files
uploaded = files.upload()  # train.csv, val.csv, test.csv seçin
print(sorted(uploaded))

In [ ]:
import json, random
import numpy as np, pandas as pd, torch
from sklearn.metrics import precision_recall_fscore_support, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 491
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL = "dbmdz/bert-base-turkish-cased"
MAX_LENGTH, BATCH, EPOCHS, LR = 256, 16, 8, 3e-5
device = "cuda" if torch.cuda.is_available() else "cpu"

train, val, test = (pd.read_csv(f"{s}.csv") for s in ("train", "val", "test"))
LABELS = [c for c in train.columns if c not in ("review_id", "text")]
assert list(val.columns) == list(train.columns) == list(test.columns)
print("cihaz:", device, "| etiketler:", LABELS)
print("yorum:", len(train), len(val), len(test))
print("train etiket sayıları:", train[LABELS].sum().to_dict())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def batches(df, shuffle):
    order = np.random.permutation(len(df)) if shuffle else np.arange(len(df))
    for start in range(0, len(df), BATCH):
        part = df.iloc[order[start:start + BATCH]]
        enc = tokenizer(part["text"].astype(str).tolist(), padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
        yield enc, torch.tensor(part[LABELS].to_numpy(), dtype=torch.float32, device=device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), problem_type="multi_label_classification",
    id2label=dict(enumerate(LABELS)), label2id={n: i for i, n in enumerate(LABELS)},
).to(device)

# Nadir etiketler için ağırlık: negatif / pozitif örnek oranı (üst sınır 20)
pos = train[LABELS].sum().to_numpy()
pos_weight = torch.tensor(np.clip((len(train) - pos) / np.maximum(pos, 1), 1, 20),
                          dtype=torch.float32, device=device)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * steps), steps)

In [ ]:
def probabilities(df):
    model.eval(); out = []
    with torch.no_grad():
        for enc, _ in batches(df, shuffle=False):
            out.append(torch.sigmoid(model(**enc).logits).cpu().numpy())
    return np.vstack(out)

best, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train(); total = 0.0
    for enc, y in batches(train, shuffle=True):
        loss = loss_fn(model(**enc).logits, y)
        loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item()
    p = probabilities(val)
    micro = f1_score(val[LABELS].to_numpy(), p >= 0.5, average="micro", zero_division=0)
    print(f"epoch {epoch}  kayıp {total:.3f}  val micro-F1 {micro:.3f}")
    if micro > best:
        best, best_state = micro, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print("seçilen val micro-F1:", round(best, 3))

In [ ]:
# Etiket başına eşik: val'de F1'i en yüksek yapan değer (val'de örneği olmayan etiket 0.5 kalır)
GRID = np.round(np.arange(0.1, 0.91, 0.05), 2)
p_val, y_val = probabilities(val), val[LABELS].to_numpy()
thresholds = {}
for j, name in enumerate(LABELS):
    if y_val[:, j].sum():
        scores = [f1_score(y_val[:, j], p_val[:, j] >= t, zero_division=0) for t in GRID]
        thresholds[name] = float(GRID[int(np.argmax(scores))])
print(thresholds)

## Test (bir kez)

Sonuçları kopyalayıp sohbete yapıştırın; rapordaki karşılaştırma tablosuna girer.

In [ ]:
p_test, y_test = probabilities(test), test[LABELS].to_numpy()
cut = np.array([thresholds.get(n, 0.5) for n in LABELS])
pred = (p_test >= cut).astype(int)
prec, rec, f1, sup = precision_recall_fscore_support(y_test, pred, zero_division=0)
table = pd.DataFrame({"label": LABELS, "precision": prec, "recall": rec, "f1": f1, "support": sup})
for avg in ("micro", "macro"):
    p_, r_, f_, _ = precision_recall_fscore_support(y_test, pred, average=avg, zero_division=0)
    table.loc[len(table)] = [f"[{avg}]", p_, r_, f_, int(y_test.sum())]
print(table.to_string(index=False, float_format="%.3f"))

In [ ]:
import shutil
model.save_pretrained("berturk_v0"); tokenizer.save_pretrained("berturk_v0")
json.dump(thresholds, open("berturk_v0/thresholds.json", "w"), indent=2)
json.dump({"val_micro_f1": best, "epochs": EPOCHS, "lr": LR, "max_length": MAX_LENGTH,
           "seed": SEED, "train": len(train), "val": len(val), "test": len(test)},
          open("berturk_v0/training.json", "w"), indent=2)
shutil.make_archive("berturk_v0", "zip", "berturk_v0")
files.download("berturk_v0.zip")